In [ ]:
# ======================================================================
# Taller de Bases de Datos — Prototipo ejecutable (Google Colab)
# **Universidad ECCI · Ingeniería de Requerimientos**
#
# Equipo: Cristian Andrés Romero Gómez · Wilmer Alejandro Londoño Martínez · Axel
#
# Motor: **SQLite** (vía el módulo `sqlite3` de Python, no requiere instalación adicional en Colab)
#
# Este notebook crea, puebla y prueba la base de datos de:
#
# - **Caso 1 — Sistema de citas médicas de una EPS**
#
# siguiendo el DDL, las reglas de negocio, las vistas, las consultas (4.7) y las pruebas de integridad/concurrencia (4.8) definidas en el expediente del taller.
#
# > No se usa ningún dato personal real: todos los nombres y valores de prueba son inventados, tal como exige la política de IA del taller.
# ======================================================================

# Celda 0 — Librerías necesarias
# Todas vienen preinstaladas en Google Colab, no hace falta pip install.
import sqlite3
import pandas as pd
import random
import threading
from datetime import datetime, timedelta

# `display` existe en Colab/Jupyter; si se corre como script normal (VS Code, terminal) se emula con print.
try:
    display
except NameError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("Librerías cargadas: sqlite3", sqlite3.sqlite_version, "| pandas", pd.__version__)

# ======================================================================
# # CASO 1 · Sistema de citas médicas de una EPS
#
# Se crea la base `caso1_citas.db` con las tablas `paciente`, `medico`, `disponibilidad`, `cita`,
# la restricción `UNIQUE (id_medico, fecha, hora)` que implementa RN-01, y las 2 vistas exigidas.
# ======================================================================

# 1.1 Conexión y DDL — Caso 1 (idéntico al script de la sección 4.6 del expediente)
conn1 = sqlite3.connect('caso1_citas.db')
conn1.execute("PRAGMA foreign_keys = ON;")
cur1 = conn1.cursor()

cur1.executescript('''
DROP VIEW IF EXISTS vista_historial_paciente;
DROP VIEW IF EXISTS vista_agenda_medico;
DROP TABLE IF EXISTS cita;
DROP TABLE IF EXISTS disponibilidad;
DROP TABLE IF EXISTS medico;
DROP TABLE IF EXISTS paciente;

CREATE TABLE paciente (
  id_paciente INTEGER PRIMARY KEY,
  nombre TEXT NOT NULL,
  documento TEXT NOT NULL UNIQUE,
  fecha_nacimiento DATE NOT NULL
);

CREATE TABLE medico (
  id_medico INTEGER PRIMARY KEY,
  nombre TEXT NOT NULL,
  especialidad TEXT NOT NULL
);

CREATE TABLE disponibilidad (
  id_disponibilidad INTEGER PRIMARY KEY,
  id_medico INTEGER NOT NULL REFERENCES medico(id_medico),
  fecha DATE NOT NULL,
  hora_inicio TEXT NOT NULL,
  hora_fin TEXT NOT NULL
);

CREATE TABLE cita (
  id_cita INTEGER PRIMARY KEY,
  id_paciente INTEGER NOT NULL REFERENCES paciente(id_paciente),
  id_medico INTEGER NOT NULL REFERENCES medico(id_medico),
  fecha DATE NOT NULL,
  hora TEXT NOT NULL,
  estado TEXT NOT NULL CHECK (estado IN ('activa','cancelada','completada')),
  motivo_cancelacion TEXT,
  UNIQUE (id_medico, fecha, hora)
);

CREATE VIEW vista_agenda_medico AS
  SELECT m.nombre AS medico, c.fecha, c.hora, p.nombre AS paciente, c.estado
  FROM cita c JOIN medico m ON c.id_medico = m.id_medico
  JOIN paciente p ON c.id_paciente = p.id_paciente
  WHERE c.estado = 'activa';

CREATE VIEW vista_historial_paciente AS
  SELECT p.id_paciente, p.nombre AS paciente, c.fecha, c.hora,
         m.nombre AS medico, c.estado
  FROM cita c
  JOIN paciente p ON c.id_paciente = p.id_paciente
  JOIN medico m ON c.id_medico = m.id_medico
  ORDER BY p.id_paciente, c.fecha DESC;

CREATE INDEX idx_cita_medico_fecha ON cita (id_medico, fecha);
CREATE INDEX idx_cita_paciente ON cita (id_paciente);
CREATE INDEX idx_cita_estado_fecha ON cita (estado, fecha);
''')
conn1.commit()
print("Caso 1: tablas, vistas e índices creados en caso1_citas.db")

# 1.2 Datos de prueba (inventados) — pacientes, médicos, disponibilidad y citas
pacientes = [
    (1, "Laura Gómez", "CC1001", "1990-04-12"),
    (2, "Andrés Pérez", "CC1002", "1985-11-02"),
    (3, "María Torres", "CC1003", "2001-01-20"),
    (4, "Julián Ruiz", "CC1004", "1978-07-08"),
    (5, "Camila Rey", "CC1005", "1995-09-30"),
]
medicos = [
    (1, "Dr. Sánchez", "Medicina General"),
    (2, "Dra. Ortiz", "Pediatría"),
    (3, "Dr. Vargas", "Cardiología"),
]
disponibilidad = [
    (1, 1, "2026-09-10", "08:00", "12:00"),
    (2, 2, "2026-09-10", "09:00", "13:00"),
    (3, 3, "2026-09-11", "14:00", "18:00"),
]
citas = [
    (1, 1, 1, "2026-09-10", "08:20", "activa", None),
    (2, 2, 1, "2026-09-10", "08:40", "activa", None),
    (3, 3, 2, "2026-09-10", "09:00", "activa", None),
    (4, 4, 3, "2026-09-11", "14:30", "cancelada", "El paciente no puede asistir"),
    (5, 5, 3, "2026-09-11", "15:00", "activa", None),
]

cur1.executemany("INSERT INTO paciente VALUES (?,?,?,?)", pacientes)
cur1.executemany("INSERT INTO medico VALUES (?,?,?)", medicos)
cur1.executemany("INSERT INTO disponibilidad VALUES (?,?,?,?,?)", disponibilidad)
cur1.executemany("INSERT INTO cita VALUES (?,?,?,?,?,?,?)", citas)
conn1.commit()
print(f"Insertados: {len(pacientes)} pacientes, {len(medicos)} médicos, "
      f"{len(disponibilidad)} franjas, {len(citas)} citas")

# 1.3 Prueba de integridad I-01 (RN-01): un médico no puede tener dos citas activas
# en la misma franja horaria -> debe fallar por la restricción UNIQUE (id_medico, fecha, hora)
try:
    cur1.execute("""INSERT INTO cita (id_cita, id_paciente, id_medico, fecha, hora, estado)
                     VALUES (99, 3, 1, '2026-09-10', '08:20', 'activa')""")
    conn1.commit()
    print("ERROR: se permitió una cita duplicada (esto NO debería pasar)")
except sqlite3.IntegrityError as e:
    print("OK — la base rechazó la cita duplicada como se esperaba:")
    print(" ", e)

# 1.4 Prueba I-02 (RN-03): no reservar una franja que ya pasó
# Aquí se valida en la capa de aplicación (Python), tal como indica el expediente (regla de negocio, no solo DDL)
def reservar_cita(cur, id_cita, id_paciente, id_medico, fecha_hora_dt, estado="activa"):
    if fecha_hora_dt < datetime.now():
        raise ValueError("No se puede reservar una franja que ya pasó (RN-03)")
    fecha = fecha_hora_dt.strftime("%Y-%m-%d")
    hora = fecha_hora_dt.strftime("%H:%M")
    cur.execute("""INSERT INTO cita (id_cita, id_paciente, id_medico, fecha, hora, estado)
                    VALUES (?,?,?,?,?,?)""", (id_cita, id_paciente, id_medico, fecha, hora, estado))

try:
    reservar_cita(cur1, 100, 2, 2, datetime(2020, 1, 1, 9, 0))
    print("ERROR: se permitió reservar una franja pasada (no debería pasar)")
except ValueError as e:
    print("OK —", e)

# Prueba positiva: una franja futura SÍ se permite (se hace rollback para no alterar los datos de prueba)
manana = (datetime.now() + timedelta(days=1)).replace(hour=9, minute=0, second=0, microsecond=0)
reservar_cita(cur1, 101, 2, 2, manana)
conn1.rollback()
print("OK — una franja futura se acepta (RN-03); se revirtió para no ensuciar los datos")

# 1.5 Consultas operativas Q-01 a Q-08 del expediente (sección 4.7)

print("Q-03 · Agenda del Dr. Sánchez el 2026-09-10:")
display(pd.read_sql_query(
    "SELECT * FROM vista_agenda_medico WHERE medico='Dr. Sánchez' AND fecha='2026-09-10'", conn1))

print("Q-04 · Historial de citas de Laura Gómez:")
display(pd.read_sql_query(
    "SELECT * FROM vista_historial_paciente WHERE paciente='Laura Gómez'", conn1))

print("Q-05 · Citas canceladas por mes:")
display(pd.read_sql_query(
    """SELECT strftime('%Y-%m', fecha) AS mes, COUNT(*) AS canceladas
       FROM cita WHERE estado='cancelada' GROUP BY mes""", conn1))

print("Q-06 · Médico con más citas programadas (activas):")
display(pd.read_sql_query(
    """SELECT m.nombre, COUNT(*) AS citas
       FROM cita c JOIN medico m ON c.id_medico = m.id_medico
       WHERE c.estado='activa' GROUP BY m.nombre ORDER BY citas DESC""", conn1))

print("Q-08 · Excepción: franjas con más de una cita activa (debe salir vacío si todo está bien)")
display(pd.read_sql_query(
    """SELECT id_medico, fecha, hora, COUNT(*) AS n
       FROM cita WHERE estado='activa'
       GROUP BY id_medico, fecha, hora HAVING COUNT(*) > 1""", conn1))

# 1.6 Rendimiento: EXPLAIN QUERY PLAN sobre la consulta Q-03 (evidencia de uso del índice)
plan = cur1.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM cita WHERE id_medico = 1 AND fecha = '2026-09-10'"
).fetchall()
for fila in plan:
    print(fila)

# 1.7 Prueba de concurrencia real: dos hilos, cada uno con su propia conexión, intentan reservar
# la MISMA franja al mismo tiempo (una barrera los suelta a la vez).
# Se activa WAL y un timeout para que el segundo escritor espere el bloqueo en vez de fallar con "database is locked".
conn1.commit()
conn1.execute("PRAGMA journal_mode=WAL;")

resultados = {}
barrera = threading.Barrier(2)

def intentar_reservar(nombre, id_cita):
    conn = sqlite3.connect('caso1_citas.db', timeout=10)   # conexión propia por hilo
    try:
        barrera.wait()                                      # ambos arrancan al mismo tiempo
        conn.execute("""INSERT INTO cita (id_cita, id_paciente, id_medico, fecha, hora, estado)
                         VALUES (?, 4, 2, '2026-09-12', '10:00', 'activa')""", (id_cita,))
        conn.commit()
        resultados[nombre] = True
    except sqlite3.IntegrityError:
        conn.rollback()
        resultados[nombre] = False
    finally:
        conn.close()

hilos = [threading.Thread(target=intentar_reservar, args=("A", 201)),
         threading.Thread(target=intentar_reservar, args=("B", 202))]
for h in hilos: h.start()
for h in hilos: h.join()

print(f"Conexión A reservó la franja: {resultados['A']}")
print(f"Conexión B reservó la franja: {resultados['B']}")
assert sum(resultados.values()) == 1, "Exactamente una de las dos debía tener éxito"
print("OK — la restricción UNIQUE resolvió la concurrencia como exige RN-01 (gana una sola, sin importar cuál)")

# ======================================================================
# ## Caso 1 · Ampliación: JOINs y consultas avanzadas
#
# Esta sección amplía el prototipo con:
#
# - **Más datos de prueba** (historial de agosto, un paciente y un médico sin citas) para que los JOINs muestren casos interesantes.
# - **JOINs**: `INNER`, `LEFT`, *self join*, `CROSS JOIN` y *anti-join* (`NOT EXISTS`).
# - **Consultas avanzadas**: CTE (`WITH`), subconsultas, funciones de ventana (`RANK`, `LAG`, `ROW_NUMBER`, acumulados), `CASE`, `HAVING` y una vista nueva.
#
# > Todos los datos siguen siendo inventados.
# ======================================================================

# 1.8 Datos adicionales para probar JOINs (idempotente: INSERT OR IGNORE)
pacientes_extra = [
    (6, "Sofía Díaz", "CC1006", "1960-03-15"),
    (7, "Tomás Herrera", "CC1007", "2015-06-01"),
    (8, "Valentina Cruz", "CC1008", "1999-12-25"),   # sin citas a propósito (para LEFT JOIN)
]
medicos_extra = [
    (4, "Dra. Mejía", "Dermatología"),                # sin citas a propósito
]
disponibilidad_extra = [
    (4, 1, "2026-09-12", "08:00", "10:00"),
    (5, 3, "2026-08-12", "14:00", "18:00"),
    (6, 1, "2026-08-05", "08:00", "12:00"),
    (7, 2, "2026-09-12", "09:00", "12:00"),   # cubre la cita de la prueba de concurrencia (1.7)
]
citas_extra = [
    (10, 1, 1, "2026-08-05", "08:00", "completada", None),
    (11, 1, 3, "2026-08-12", "14:00", "completada", None),
    (12, 2, 1, "2026-08-05", "08:20", "completada", None),
    (13, 2, 1, "2026-08-19", "09:00", "completada", None),
    (14, 3, 2, "2026-08-20", "09:00", "completada", None),
    (15, 4, 3, "2026-08-12", "14:20", "completada", None),
    (16, 4, 1, "2026-08-19", "10:00", "cancelada", "Cruce con horario laboral"),
    (17, 5, 3, "2026-08-12", "14:40", "completada", None),
    (18, 5, 1, "2026-08-26", "08:00", "cancelada", "Enfermedad"),
    (19, 7, 2, "2026-09-10", "09:20", "activa", None),
    (20, 6, 3, "2026-09-11", "15:20", "activa", None),
    (21, 6, 1, "2026-09-10", "09:00", "activa", None),
    # Cita registrada FUERA de la franja de disponibilidad del médico 1 (08:00-12:00): se detecta en J-05
    (301, 5, 1, "2026-09-10", "13:00", "activa", None),
]

cur1.executemany("INSERT OR IGNORE INTO paciente VALUES (?,?,?,?)", pacientes_extra)
cur1.executemany("INSERT OR IGNORE INTO medico VALUES (?,?,?)", medicos_extra)
cur1.executemany("INSERT OR IGNORE INTO disponibilidad VALUES (?,?,?,?,?)", disponibilidad_extra)
cur1.executemany("INSERT OR IGNORE INTO cita VALUES (?,?,?,?,?,?,?)", citas_extra)
conn1.commit()

for tabla in ("paciente", "medico", "disponibilidad", "cita"):
    n = cur1.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"{tabla}: {n} filas")

# Helper para ejecutar y mostrar una consulta con su título
def consulta(conn, titulo, sql):
    print(titulo)
    df = pd.read_sql_query(sql, conn)
    display(df)
    return df

# ======================================================================
# ### JOINs
# ======================================================================

# J-01 · INNER JOIN de 3 tablas: detalle completo de cada cita (con edad calculada)
_ = consulta(conn1, "J-01 · Detalle de citas (cita + paciente + médico):", '''
SELECT c.id_cita, c.fecha, c.hora,
       p.nombre AS paciente,
       CAST((julianday(c.fecha) - julianday(p.fecha_nacimiento)) / 365.25 AS INTEGER) AS edad,
       m.nombre AS medico, m.especialidad, c.estado
FROM cita c
INNER JOIN paciente p ON p.id_paciente = c.id_paciente
INNER JOIN medico   m ON m.id_medico   = c.id_medico
ORDER BY c.fecha, c.hora
''')

# J-02 · LEFT JOIN: entidades "huérfanas" (sin ninguna cita)
_ = consulta(conn1, "J-02a · Pacientes que nunca han tenido una cita:", '''
SELECT p.id_paciente, p.nombre, p.documento
FROM paciente p
LEFT JOIN cita c ON c.id_paciente = p.id_paciente
WHERE c.id_cita IS NULL
''')

_ = consulta(conn1, "J-02b · Médicos sin citas asignadas:", '''
SELECT m.id_medico, m.nombre, m.especialidad
FROM medico m
LEFT JOIN cita c ON c.id_medico = m.id_medico
WHERE c.id_cita IS NULL
''')

# J-03 · LEFT JOIN con condición de rango: ocupación de cada franja de disponibilidad
# Se asume que cada cita dura 20 minutos, por lo que cupos = minutos de la franja / 20
_ = consulta(conn1, "J-03 · Ocupación de cada franja de disponibilidad (citas activas):", '''
WITH franja AS (
  SELECT d.id_disponibilidad, m.nombre AS medico, d.fecha, d.hora_inicio, d.hora_fin,
         ( (CAST(substr(d.hora_fin,1,2)    AS INTEGER)*60 + CAST(substr(d.hora_fin,4,2)    AS INTEGER))
         - (CAST(substr(d.hora_inicio,1,2) AS INTEGER)*60 + CAST(substr(d.hora_inicio,4,2) AS INTEGER)) ) / 20
         AS cupos_totales
  FROM disponibilidad d
  JOIN medico m ON m.id_medico = d.id_medico
)
SELECT f.medico, f.fecha, f.hora_inicio, f.hora_fin, f.cupos_totales,
       COUNT(c.id_cita) AS cupos_ocupados,
       ROUND(100.0 * COUNT(c.id_cita) / f.cupos_totales, 1) AS pct_ocupacion
FROM franja f
JOIN disponibilidad d ON d.id_disponibilidad = f.id_disponibilidad
LEFT JOIN cita c ON c.id_medico = d.id_medico
                AND c.fecha = d.fecha
                AND c.hora >= d.hora_inicio AND c.hora < d.hora_fin
                AND c.estado = 'activa'
GROUP BY f.id_disponibilidad
ORDER BY f.fecha, f.medico
''')

# J-04 · SELF JOIN: pares de pacientes con cita activa con el mismo médico el mismo día
_ = consulta(conn1, "J-04 · Pacientes que coinciden con el mismo médico el mismo día (self join):", '''
SELECT c1.fecha, m.nombre AS medico,
       p1.nombre AS paciente_1, c1.hora AS hora_1,
       p2.nombre AS paciente_2, c2.hora AS hora_2
FROM cita c1
JOIN cita c2     ON c1.id_medico = c2.id_medico AND c1.fecha = c2.fecha AND c1.id_cita < c2.id_cita
JOIN paciente p1 ON p1.id_paciente = c1.id_paciente
JOIN paciente p2 ON p2.id_paciente = c2.id_paciente
JOIN medico m    ON m.id_medico = c1.id_medico
WHERE c1.estado = 'activa' AND c2.estado = 'activa'
ORDER BY c1.fecha, m.nombre, c1.hora
''')

# J-05 · ANTI-JOIN (NOT EXISTS): citas activas que NO caen dentro de ninguna franja de disponibilidad
# Es una consulta de excepción: en producción debe salir vacía. Aquí sale SOLO la cita 301, sembrada a propósito.
_ = consulta(conn1, "J-05 · Excepción: citas activas fuera de la disponibilidad del médico:", '''
SELECT c.id_cita, m.nombre AS medico, c.fecha, c.hora
FROM cita c
JOIN medico m ON m.id_medico = c.id_medico
WHERE c.estado = 'activa'
  AND NOT EXISTS (
    SELECT 1 FROM disponibilidad d
    WHERE d.id_medico = c.id_medico AND d.fecha = c.fecha
      AND c.hora >= d.hora_inicio AND c.hora < d.hora_fin
  )
ORDER BY c.fecha, c.hora
''')

# J-06 · CROSS JOIN + LEFT JOIN: matriz especialidad x estado, incluyendo combinaciones con 0 citas
_ = consulta(conn1, "J-06 · Matriz especialidad x estado (rellena con ceros):", '''
WITH estados(estado) AS (VALUES ('activa'), ('completada'), ('cancelada')),
     especialidades AS (SELECT DISTINCT especialidad FROM medico)
SELECT e.especialidad,
       SUM(CASE WHEN s.estado = 'activa'     THEN COALESCE(x.n, 0) END) AS activas,
       SUM(CASE WHEN s.estado = 'completada' THEN COALESCE(x.n, 0) END) AS completadas,
       SUM(CASE WHEN s.estado = 'cancelada'  THEN COALESCE(x.n, 0) END) AS canceladas
FROM especialidades e
CROSS JOIN estados s
LEFT JOIN (
    SELECT m.especialidad, c.estado, COUNT(*) AS n
    FROM cita c JOIN medico m ON m.id_medico = c.id_medico
    GROUP BY m.especialidad, c.estado
) x ON x.especialidad = e.especialidad AND x.estado = s.estado
GROUP BY e.especialidad
ORDER BY e.especialidad
''')

# ======================================================================
# ### Consultas avanzadas
# ======================================================================

# A-01 · CTE: tasa de cancelación por médico (incluye médicos sin citas con LEFT JOIN)
_ = consulta(conn1, "A-01 · Tasa de cancelación por médico (CTE):", '''
WITH resumen AS (
  SELECT id_medico,
         COUNT(*) AS total,
         SUM(estado = 'activa')     AS activas,
         SUM(estado = 'completada') AS completadas,
         SUM(estado = 'cancelada')  AS canceladas
  FROM cita
  GROUP BY id_medico
)
SELECT m.nombre AS medico, m.especialidad,
       COALESCE(r.total, 0)       AS total,
       COALESCE(r.activas, 0)     AS activas,
       COALESCE(r.completadas, 0) AS completadas,
       COALESCE(r.canceladas, 0)  AS canceladas,
       ROUND(100.0 * r.canceladas / NULLIF(r.total, 0), 1) AS pct_cancelacion
FROM medico m
LEFT JOIN resumen r ON r.id_medico = m.id_medico
ORDER BY pct_cancelacion DESC
''')

# A-02 · Subconsulta escalar en HAVING: pacientes con más citas que el promedio
_ = consulta(conn1, "A-02 · Pacientes con más citas que el promedio (subconsulta):", '''
SELECT p.nombre, COUNT(*) AS n_citas
FROM cita c JOIN paciente p ON p.id_paciente = c.id_paciente
GROUP BY p.id_paciente
HAVING COUNT(*) > (SELECT AVG(n) FROM (SELECT COUNT(*) AS n FROM cita GROUP BY id_paciente))
ORDER BY n_citas DESC
''')

# A-03 · Función de ventana RANK: ranking de médicos por mes (solo citas no canceladas)
_ = consulta(conn1, "A-03 · Ranking mensual de médicos por citas atendidas/programadas (RANK):", '''
WITH mensual AS (
  SELECT strftime('%Y-%m', c.fecha) AS mes, m.nombre AS medico, COUNT(*) AS citas
  FROM cita c JOIN medico m ON m.id_medico = c.id_medico
  WHERE c.estado <> 'cancelada'
  GROUP BY mes, m.id_medico
)
SELECT mes, medico, citas,
       RANK() OVER (PARTITION BY mes ORDER BY citas DESC) AS posicion
FROM mensual
ORDER BY mes, posicion
''')

# A-04 · LAG: días transcurridos entre citas consecutivas de cada paciente
_ = consulta(conn1, "A-04 · Días entre citas consecutivas por paciente (LAG):", '''
SELECT p.nombre AS paciente, c.fecha, m.especialidad,
       LAG(c.fecha) OVER (PARTITION BY c.id_paciente ORDER BY c.fecha, c.hora) AS cita_anterior,
       CAST(julianday(c.fecha) - julianday(
            LAG(c.fecha) OVER (PARTITION BY c.id_paciente ORDER BY c.fecha, c.hora)) AS INTEGER) AS dias_desde_anterior
FROM cita c
JOIN paciente p ON p.id_paciente = c.id_paciente
JOIN medico m   ON m.id_medico   = c.id_medico
WHERE c.estado <> 'cancelada'
ORDER BY p.nombre, c.fecha, c.hora
''')

# A-05 · Acumulado con SUM() OVER: citas por fecha y total acumulado
_ = consulta(conn1, "A-05 · Citas por fecha con total acumulado (SUM OVER):", '''
SELECT fecha,
       COUNT(*) AS citas_del_dia,
       SUM(COUNT(*)) OVER (ORDER BY fecha) AS acumulado
FROM cita
WHERE estado <> 'cancelada'
GROUP BY fecha
ORDER BY fecha
''')

# A-06 · ROW_NUMBER: última cita (no cancelada) de cada paciente -> patrón "top-1 por grupo"
_ = consulta(conn1, "A-06 · Última cita de cada paciente (ROW_NUMBER):", '''
WITH ordenadas AS (
  SELECT c.*, ROW_NUMBER() OVER (PARTITION BY c.id_paciente ORDER BY c.fecha DESC, c.hora DESC) AS rn
  FROM cita c
  WHERE c.estado <> 'cancelada'
)
SELECT p.nombre AS paciente, o.fecha, o.hora, m.nombre AS medico, o.estado
FROM ordenadas o
JOIN paciente p ON p.id_paciente = o.id_paciente
JOIN medico m   ON m.id_medico   = o.id_medico
WHERE o.rn = 1
ORDER BY p.nombre
''')

# A-07 · CASE: citas por rango etario del paciente
_ = consulta(conn1, "A-07 · Citas por rango de edad (CASE):", '''
WITH base AS (
  SELECT c.id_cita,
         CAST((julianday(c.fecha) - julianday(p.fecha_nacimiento)) / 365.25 AS INTEGER) AS edad
  FROM cita c JOIN paciente p ON p.id_paciente = c.id_paciente
)
SELECT CASE WHEN edad < 18 THEN '1. Menor de 18'
            WHEN edad < 40 THEN '2. 18 a 39'
            WHEN edad < 60 THEN '3. 40 a 59'
            ELSE '4. 60 o más' END AS rango_edad,
       COUNT(*) AS citas
FROM base
GROUP BY rango_edad
ORDER BY rango_edad
''')

# A-08 · HAVING COUNT(DISTINCT): pacientes atendidos por más de una especialidad
_ = consulta(conn1, "A-08 · Pacientes vistos por 2 o más especialidades:", '''
SELECT p.nombre AS paciente,
       COUNT(DISTINCT m.especialidad) AS especialidades,
       GROUP_CONCAT(DISTINCT m.especialidad) AS lista
FROM cita c
JOIN paciente p ON p.id_paciente = c.id_paciente
JOIN medico m   ON m.id_medico   = c.id_medico
WHERE c.estado <> 'cancelada'
GROUP BY p.id_paciente
HAVING COUNT(DISTINCT m.especialidad) >= 2
ORDER BY especialidades DESC, p.nombre
''')

# A-09 · Motivos de cancelación más frecuentes (JOIN + GROUP BY + porcentaje con ventana)
_ = consulta(conn1, "A-09 · Motivos de cancelación y su peso porcentual:", '''
SELECT motivo_cancelacion AS motivo,
       COUNT(*) AS veces,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM cita
WHERE estado = 'cancelada'
GROUP BY motivo_cancelacion
ORDER BY veces DESC
''')

# 1.9 Vista nueva basada en JOINs: resumen operativo por médico
cur1.executescript('''
DROP VIEW IF EXISTS vista_resumen_medico;
CREATE VIEW vista_resumen_medico AS
  SELECT m.id_medico, m.nombre AS medico, m.especialidad,
         COUNT(c.id_cita)                                   AS total_citas,
         COALESCE(SUM(c.estado = 'activa'), 0)              AS activas,
         COALESCE(SUM(c.estado = 'completada'), 0)          AS completadas,
         COALESCE(SUM(c.estado = 'cancelada'), 0)           AS canceladas,
         COUNT(DISTINCT c.id_paciente)                      AS pacientes_distintos
  FROM medico m
  LEFT JOIN cita c ON c.id_medico = m.id_medico
  GROUP BY m.id_medico;
''')
conn1.commit()
_ = consulta(conn1, "Vista nueva · vista_resumen_medico:", "SELECT * FROM vista_resumen_medico")

# 1.10 Rendimiento: ¿los JOINs usan los índices creados en el DDL?
print("EXPLAIN QUERY PLAN · JOIN cita + medico filtrado por médico y fecha:")
for fila in cur1.execute('''
    EXPLAIN QUERY PLAN
    SELECT c.hora, p.nombre
    FROM cita c
    JOIN paciente p ON p.id_paciente = c.id_paciente
    JOIN medico m   ON m.id_medico   = c.id_medico
    WHERE c.id_medico = 1 AND c.fecha = '2026-09-10'
''').fetchall():
    print(fila)

# ======================================================================
# ## Resumen de lo verificado en este notebook (Caso 1)
#
# | Prueba | Resultado |
# |---|---|
# | Integridad (UNIQUE) | ✅ RN-01 |
# | Regla de negocio en aplicación | ✅ RN-03 (franja pasada rechazada, futura aceptada) |
# | Consultas Q-01…Q-08 | ✅ ejecutadas |
# | Rendimiento (EXPLAIN QUERY PLAN) | ✅ |
# | Concurrencia (dos hilos, misma franja) | ✅ gana uno solo |
# | JOINs (INNER, LEFT, self, CROSS, anti-join) | ✅ J-01 a J-06 |
# | Consultas avanzadas (CTE, ventanas, subconsultas) | ✅ A-01 a A-09 |
# | Vista nueva basada en JOINs | ✅ vista_resumen_medico |
#
# El archivo `caso1_citas.db` queda en el entorno de Colab y puede descargarse desde el panel de archivos.
# ======================================================================

conn1.close()
